# FOML Project — Step 1: Verify Our Dataset

**Project:** AI-Generated vs Real Image Detection

This notebook only audits the processed dataset. It does **not** change images or train a model.

It checks:
1. The ZIP contains readable images in the expected `real/` and `ai_generated/` folders.
2. Image counts by class and subject category.
3. Image dimensions, color modes, and file formats.
4. Corrupt images and exact duplicate images, including duplicates appearing in both classes.
5. Whether `metadata.csv` agrees with the image files.
6. A small visual preview so you can catch incorrect labels or bad crops.

**Input:** the `processed_foml_dataset.zip` downloaded from our preprocessing notebook. Keep your original ZIP files unchanged.

## 1. Upload the processed dataset ZIP

Run this cell and select `processed_foml_dataset.zip` from your computer.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_files:
    raise ValueError("Please upload the processed_foml_dataset.zip file.")

# If multiple ZIP files were uploaded, choose the processed dataset explicitly.
DATASET_ZIP = next((name for name in zip_files if 'processed_foml_dataset' in name.lower()), zip_files[0])
print("Dataset ZIP:", DATASET_ZIP)

EXTRACT_DIR = Path('/content/foml_dataset_audit')
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise ValueError(f"The ZIP archive may be damaged. First problematic member: {bad_member}")
    zf.extractall(EXTRACT_DIR)

print("Extraction complete.")
print("Extracted to:", EXTRACT_DIR)

## 2. Find the dataset folders and inspect the file structure

In [ ]:
from collections import Counter, defaultdict
from PIL import Image, ImageOps, UnidentifiedImageError
import hashlib
import pandas as pd
import matplotlib.pyplot as plt

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff', '.heic', '.heif'}
LABEL_NAMES = {'real': 'real', 'ai_generated': 'ai_generated', 'ai-generated': 'ai_generated', 'ai': 'ai_generated', 'synthetic': 'ai_generated'}
CATEGORY_ALIASES = {
    'food': 'food',
    'plant': 'plants_nature', 'plants': 'plants_nature', 'nature': 'plants_nature',
    'plants_nature': 'plants_nature', 'plants_and_nature': 'plants_nature',
    'building_and_street': 'buildings_and_streets', 'buildings_and_streets': 'buildings_and_streets',
    'buildings_streets': 'buildings_and_streets', 'streets': 'buildings_and_streets',
    'household': 'household_objects', 'household_object': 'household_objects', 'household_objects': 'household_objects',
    'vehicle': 'vehicles', 'vehicles': 'vehicles',
    'animal': 'animals_and_pets', 'animals': 'animals_and_pets', 'pets': 'animals_and_pets',
    'animals_pets': 'animals_and_pets', 'animals_and_pets': 'animals_and_pets'
}
CATEGORY_ORDER = ['food', 'plants_nature', 'buildings_and_streets', 'household_objects', 'vehicles', 'animals_and_pets']

def normalize_name(value):
    return str(value).strip().lower().replace('-', '_').replace(' ', '_')

# Find metadata.csv if present; its parent should usually be the dataset root.
metadata_files = list(EXTRACT_DIR.rglob('metadata.csv'))
if metadata_files:
    DATASET_ROOT = metadata_files[0].parent
else:
    # Otherwise locate a directory that has real/ and ai_generated/ children.
    candidates = []
    for folder in [EXTRACT_DIR] + [p for p in EXTRACT_DIR.rglob('*') if p.is_dir()]:
        child_names = {p.name.lower() for p in folder.iterdir() if p.is_dir()} if folder.exists() else set()
        if 'real' in child_names and ('ai_generated' in child_names or 'ai' in child_names):
            candidates.append(folder)
    DATASET_ROOT = candidates[0] if candidates else EXTRACT_DIR

print("Dataset root:", DATASET_ROOT)
print("Metadata found:", bool(metadata_files))
print("Top-level contents:")
for item in sorted(DATASET_ROOT.iterdir()):
    print(' -', item.name, '(folder)' if item.is_dir() else '(file)')

## 3. Count the images and check that each file opens

This does not infer whether an image is truly AI-generated or real. It checks that folders and image files are structurally usable.

In [ ]:
image_rows = []
problems = []
pixel_hashes = defaultdict(list)

for path in DATASET_ROOT.rglob('*'):
    if not path.is_file() or path.suffix.lower() not in IMAGE_EXTS:
        continue
    # Ignore common operating-system metadata files.
    if '__MACOSX' in path.parts or path.name.startswith('._') or path.name == '.DS_Store':
        continue

    rel_parts = [normalize_name(p) for p in path.relative_to(DATASET_ROOT).parts]
    label = None
    category = None
    for i, part in enumerate(rel_parts[:-1]):
        if part in LABEL_NAMES:
            label = LABEL_NAMES[part]
            if i + 1 < len(rel_parts) - 1:
                category = CATEGORY_ALIASES.get(rel_parts[i + 1], rel_parts[i + 1])
            break

    if label is None:
        problems.append({'file': str(path.relative_to(DATASET_ROOT)), 'issue': 'Could not identify real/AI class folder'})
        continue

    try:
        with Image.open(path) as source:
            source.load()
            image = ImageOps.exif_transpose(source).convert('RGB')
            width, height = image.size
            if width < 16 or height < 16:
                raise ValueError(f"Image too small: {width}x{height}")
            digest = hashlib.sha256(f'{image.size}'.encode() + image.tobytes()).hexdigest()
            pixel_hashes[digest].append({'path': str(path.relative_to(DATASET_ROOT)), 'label': label, 'category': category})
            image_rows.append({
                'relative_path': str(path.relative_to(DATASET_ROOT)),
                'label': label,
                'category': category or 'unknown',
                'width': width,
                'height': height,
                'format': source.format or path.suffix.lower().lstrip('.'),
                'mode': source.mode,
                'pixel_sha256': digest,
                'bytes': path.stat().st_size
            })
    except Exception as exc:
        problems.append({'file': str(path.relative_to(DATASET_ROOT)), 'issue': f'Could not open image: {type(exc).__name__}: {exc}'})

images_df = pd.DataFrame(image_rows)
problems_df = pd.DataFrame(problems)

print('Readable images:', len(images_df))
print('Problem files:', len(problems_df))
print('')
if len(images_df):
    print('Counts by class:')
    display(images_df.groupby('label').size().rename('image_count').to_frame())
    print('Counts by class and category:')
    count_table = images_df.groupby(['label', 'category']).size().unstack(fill_value=0)
    display(count_table)
    print('Image dimensions:')
    display(images_df.groupby(['width', 'height']).size().rename('count').to_frame().reset_index())
    print('File formats:')
    display(images_df.groupby('format').size().rename('count').to_frame())
if len(problems_df):
    print('Files needing attention:')
    display(problems_df.head(50))

## 4. Look for exact duplicate images

The notebook checks the decoded image pixels. It finds exact duplicates, but it may not find two visually similar photos taken from slightly different angles.

In [ ]:
duplicate_rows = []
for digest, matches in pixel_hashes.items():
    if len(matches) > 1:
        labels = sorted(set(item['label'] for item in matches))
        duplicate_rows.append({
            'copies': len(matches),
            'crosses_classes': len(labels) > 1,
            'files': ' | '.join(item['path'] for item in matches)
        })

duplicates_df = pd.DataFrame(duplicate_rows)
print('Exact duplicate groups:', len(duplicates_df))
if len(duplicates_df):
    print('Duplicate groups that appear across real and AI classes:', int(duplicates_df['crosses_classes'].sum()))
    display(duplicates_df.head(50))
else:
    print('No exact duplicate pixel images found.')

## 5. Check metadata consistency

The metadata CSV may include extra columns. We check whether its image count and paths agree with the files we found.

In [ ]:
metadata_path = DATASET_ROOT / 'metadata.csv'
metadata_df = None
metadata_issues = []

if metadata_path.exists():
    try:
        metadata_df = pd.read_csv(metadata_path)
        print('Metadata rows:', len(metadata_df))
        print('Metadata columns:', list(metadata_df.columns))
        if 'label' in metadata_df.columns:
            print('Metadata labels:')
            display(metadata_df['label'].value_counts(dropna=False).to_frame('count'))
        if 'relative_path' in metadata_df.columns:
            actual_paths = set(images_df['relative_path']) if len(images_df) else set()
            metadata_paths = set(metadata_df['relative_path'].dropna().astype(str))
            # The original metadata paths may include the outer folder name; try exact and suffix matching.
            def path_matches(meta_path, actual_set):
                if meta_path in actual_set:
                    return True
                suffix = '/' + meta_path.replace('\\', '/').split('/', 1)[-1]
                return any(p.endswith(suffix) for p in actual_set)
            missing_meta_files = [p for p in metadata_paths if not path_matches(p, actual_paths)]
            print('Metadata paths not matched to scanned files:', len(missing_meta_files))
            if missing_meta_files:
                metadata_issues.extend([{'issue': 'Metadata path not matched', 'details': p} for p in missing_meta_files[:50]])
    except Exception as exc:
        metadata_issues.append({'issue': 'Could not read metadata.csv', 'details': str(exc)})
        print('Could not read metadata:', exc)
else:
    print('No metadata.csv found. The image folders can still be audited, but we should retain metadata for reproducibility.')

if metadata_issues:
    display(pd.DataFrame(metadata_issues))

## 6. Preview sample images

Use this grid to check visually for incorrect labels, strange crops, empty images, or a large visual difference between classes.

In [ ]:
if len(images_df):
    display_rows = []
    for label in ['real', 'ai_generated']:
        subset = images_df[images_df['label'] == label]
        for category in CATEGORY_ORDER:
            group = subset[subset['category'] == category]
            if len(group):
                display_rows.extend(group.head(2).to_dict('records'))

    n = len(display_rows)
    if n:
        fig, axes = plt.subplots((n + 3) // 4, 4, figsize=(14, 3 * ((n + 3) // 4)))
        axes = list(axes.flat) if hasattr(axes, 'flat') else [axes]
        for ax, row in zip(axes, display_rows):
            path = DATASET_ROOT / row['relative_path']
            with Image.open(path) as im:
                ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
            ax.set_title(f"{row['label']} | {row['category']}", fontsize=9)
            ax.axis('off')
        for ax in axes[len(display_rows):]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('No readable images to preview.')

## 7. Produce a simple audit summary

Review this summary before training. We should not proceed if an image is unreadable, a duplicate crosses class labels, or category counts are mismatched unexpectedly.

In [ ]:
summary = {
    'readable_image_count': int(len(images_df)),
    'real_count': int((images_df['label'] == 'real').sum()) if len(images_df) else 0,
    'ai_generated_count': int((images_df['label'] == 'ai_generated').sum()) if len(images_df) else 0,
    'problem_file_count': int(len(problems_df)),
    'exact_duplicate_groups': int(len(duplicates_df)),
    'cross_class_duplicate_groups': int(duplicates_df['crosses_classes'].sum()) if len(duplicates_df) else 0,
    'metadata_found': bool(metadata_path.exists()),
}
print('AUDIT SUMMARY')
for key, value in summary.items():
    print(f'{key}: {value}')

if len(images_df):
    per_category = images_df.groupby(['label', 'category']).size().unstack(fill_value=0)
    print('\nCategory balance table:')
    display(per_category)
    if set(images_df['label'].unique()) != {'real', 'ai_generated'}:
        print('WARNING: Expected both labels: real and ai_generated.')
    if len(problems_df):
        print('WARNING: Inspect problem files before training.')
    if len(duplicates_df) and duplicates_df['crosses_classes'].any():
        print('WARNING: An exact same image appears in both classes. Resolve this before training.')
    if 'category' in images_df.columns:
        for category in CATEGORY_ORDER:
            real_n = int(((images_df['label'] == 'real') & (images_df['category'] == category)).sum())
            ai_n = int(((images_df['label'] == 'ai_generated') & (images_df['category'] == category)).sum())
            if real_n != ai_n:
                print(f'NOTE: Category balance differs for {category}: real={real_n}, AI={ai_n}.')

# Save reports so the results can be shared with the team.
images_df.to_csv('/content/image_audit_report.csv', index=False)
if len(problems_df):
    problems_df.to_csv('/content/image_problems.csv', index=False)
if len(duplicates_df):
    duplicates_df.to_csv('/content/duplicate_image_groups.csv', index=False)
print('\nSaved /content/image_audit_report.csv')
print('This audit has not modified or deleted any images.')